<a href="https://colab.research.google.com/github/hamza26410/City-Library-After-School-Program-Project/blob/main/City%20Library%20After%20School%20Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [703]:
# Name: Hamza Ahmed Sayed Hassan AboZaid(حمزة أحمد سيد حسن أبو زيد)
# My ID Number: 31004260105918
# The Name Of The Project: 31004260105918_City Library After-School Program(31004260105918_City Library After-School Program.ipynb)(مشروع تخرج شامل (Capstone Project))

# **Setup and Dataset preparation:**

In [704]:
# 1.Load the dataset
import sqlite3
import pandas as pd
import json
from bs4 import BeautifulSoup

# **Task (1): Data Gathering and Combination: Part(1):**

In [705]:
# 2.connect to the database file
conn = sqlite3.connect("level 3 final project library.db")

In [706]:

# 3.(read)get members table from the database
members_df = pd.read_sql_query("SELECT * FROM members", conn)

# 4.(get)join checkouts and members directly using SQL LEFT JOIN and read the query in the dataframe
sql_query = """
SELECT
    checkouts.checkout_id,
    checkouts.member_id,
    checkouts.book_id,
    checkouts.checkout_date,
    checkouts.return_date,
    members.neighborhood,
    members.membership_status
FROM checkouts
LEFT JOIN members ON checkouts.member_id = members.member_id
"""

stage1_df = pd.read_sql_query(sql_query, conn)

In [707]:
# 5. close database connection after getting the data
conn.close()

In [708]:
# 6.(read)load the books json catalog file
books_df = pd.read_json("level 3 final project book catalog.json")

# 7.merge book details into our current data
stage2_df = pd.merge(stage1_df, books_df, on="book_id", how="left")

In [709]:
# 8.read the html file for event signups
web_tables = pd.read_html("level 3 final project event sign up.html")
web_df = web_tables[0]

# 9.rename columns so they match our main data
web_df.columns = ["member_id", "book_id", "checkout_date"]

In [710]:
# 10.create custom ids for web checkouts and set return date to empty
web_df["checkout_id"] = "WEB_" + (web_df.index + 1).astype(str)
web_df["return_date"] = None

In [711]:
# 11.add member and book details to web signups
web_enriched = pd.merge(web_df, members_df, on="member_id", how="left")
web_enriched = pd.merge(web_enriched, books_df, on="book_id", how="left")

In [712]:
# 12.Fix duplicate columns and reset indexes
stage2_df = stage2_df.loc[:, ~stage2_df.columns.duplicated()].reset_index(drop=True)
web_enriched = web_enriched.loc[:, ~web_enriched.columns.duplicated()].reset_index(drop=True)

# 13. combine all data sources into one final table
final_df = pd.concat([stage2_df, web_enriched], ignore_index=True)

# Standardize date format in final_df
final_df['checkout_date'] = pd.to_datetime(final_df['checkout_date'], errors='coerce')
final_df['return_date'] = pd.to_datetime(final_df['return_date'], errors='coerce')

# 14. save the combined dataset to csv file
final_df.to_csv("31004260105918-Library-Task(1)-Combined-Data.csv", index=False)

# 15. check the result
final_df.head()

,checkout_id,member_id,book_id,checkout_date,return_date,neighborhood,membership_status,genre,pages,publication_year,publisher,first_name,last_name,grade,join_date
0,9263,1047,517,2024-10-21,2024-11-07,Heliopolis,Inactive,Mystery,338,2015.0,Delta House,NaN,NaN,NaN,NaN
1,9340,1072,513,2025-08-24,2025-09-01,Zamalek,Active,Science,294,2021.0,Oasis Books,NaN,NaN,NaN,NaN
2,9231,1053,523,2024-02-04,2024-02-16,Heliopolis,Active,Historical,276,2018.0,Oasis Books,NaN,NaN,NaN,NaN
3,9129,1032,513,2025-06-21,2025-06-29,Nasr City,Active,Science,294,2021.0,Oasis Books,NaN,NaN,NaN,NaN
4,9370,1079,511,2025-11-11,2025-12-03,Shubra,Active,Historical,117,2016.0,Nile Press,NaN,NaN,NaN,NaN


# **Task (1): Data Gathering and Combination: Part(2):**

In [713]:
# 16.Connect to the database
conn = sqlite3.connect("level 3 final project library.db")

# 17.Question (1): Get the total number of checkouts for each member (including members with zero checkouts)
# 18.Answer (1): Use LEFT JOIN to show all members even if they have 0 checkouts
sql_query_q1 = """
SELECT
    m.member_id,
    m.first_name || ' ' || m.last_name AS name,
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, name;
"""

# 19. Run query and show result
q1_df = pd.read_sql_query(sql_query_q1, conn)
q1_df

,member_id,name,total_checkouts
0,1001,Salma Ibrahim,1
1,1002,Fares Saleh,2
2,1003,Bassel Hegazy,9
3,1004,Fares Wahba,0
4,1005,Youssef Halim,3
...,...,...,...
75,1076,Dina Wahba,7
76,1077,Lina Rashad,6
77,1078,Habiba Osman,0
78,1079,Rana Osman,10


In [714]:
# 20.Question (2): Get book titles where the author's name starts with 'A' (showing the selected letter)
# 21.Answer (2): Use WHERE with LIKE 'A%' and add a constant column for the letter
sql_query_q2 = """
SELECT
    title,
    author,
    'A' AS selected_letter
FROM books
WHERE author LIKE 'A%'
"""
# 21.Run query and show result
q2_df = pd.read_sql_query(sql_query_q2, conn)
q2_df

,title,author,selected_letter
0,The Silver Kite,Amina Darwish,A
1,Desert Compass,Amina Darwish,A
2,The Lantern Maker,Adel Roushdy,A
3,Rooftop Astronomers,Adel Roushdy,A
4,Letters to the Nile,Aya Hafez,A
5,The Paper Boat Club,Aya Hafez,A


In [715]:

# 21.Question (3): Get the top 5 most checked-out books and their checkout count
# 22.Answer (3): JOIN books with checkouts, GROUP BY book, ORDER BY count DESC, LIMIT 5
sql_query_q3 = """
SELECT
    b.book_id,
    b.title,
    COUNT(c.checkout_id) AS checkout_count
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id, b.title
ORDER BY checkout_count DESC
LIMIT 5;
"""

# 23.Run query and show result
q3_df = pd.read_sql_query(sql_query_q3, conn)
q3_df

,book_id,title,checkout_count
0,501,The Silver Kite,57
1,507,Fossils and Fireflies,55
2,513,Circuits for Beginners,46
3,519,Kites Over Cairo,38
4,525,Storms and Sailboats,25


In [716]:
# 24.Question (4): Get top 10 members who checked out the most books
# 25.Answer (4): JOIN members with checkouts, GROUP BY member, ORDER BY total DESC, LIMIT 10

conn = sqlite3.connect("level 3 final project library.db")

sql_query_q4 = sql_query_q4 = """
SELECT
    m.member_id,
    m.first_name || ' ' || m.last_name AS name,
    COUNT(c.checkout_id) AS total_checkouts
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY total_checkouts DESC
LIMIT 10;
"""
# 26.Run query and show result
q4_df = pd.read_sql_query(sql_query_q4, conn)
q4_df

,member_id,name,total_checkouts
0,1034,Aya Wahba,25
1,1044,Sherif Saleh,21
2,1008,Ziad Saleh,19
3,1010,Nour Nabil,18
4,1027,Mostafa Fouad,18
5,1018,Ahmed Shafik,17
6,1024,Youssef Hegazy,17
7,1065,Adam Fahmy,17
8,1030,Reem Osman,16
9,1047,Sara Rashad,16


In [717]:
# 27.Question (5): Get checkouts for 'Downtown' neighborhood (skip first 10, get next 10)
# 28.Answer (5): JOIN checkouts with members, filter by neighborhood, ORDER BY date DESC, LIMIT 10 OFFSET 10
sql_query_q5 = """
SELECT
    c.*,
    m.neighborhood,
    'Downtown' AS selected_neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE LOWER(m.neighborhood) = 'downtown'
ORDER BY c.checkout_date DESC
LIMIT 10 OFFSET 10
"""

# 29.Run query and show result
q5_df = pd.read_sql_query(sql_query_q5, conn)
q5_df

,checkout_id,member_id,book_id,checkout_date,return_date,neighborhood,selected_neighborhood


In [718]:
# 29.Close the database connection
conn.close()

In [719]:

# 30.Prepare the full text content
# 30.Prepare the full text content
text_content = """
==================================================
LIBRARY DATABASE SQL QUERIES REPORT - TASK 1
==================================================

Question (1): Get the total number of checkouts for each member (including members with zero checkouts).
Answer (1): LEFT JOIN members with checkouts, GROUP BY member_id
SQL Query:
SELECT
    m.member_id,
    m.first_name || ' ' || m.last_name AS name,
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, name;

Result Output:
member_id             name  total_checkouts
        1    Alice Johnson                5
        2        Bob Smith                0
        3    Charlie Brown               12


Question (2): Get book titles where the author's name starts with 'A' (showing the selected letter).
Answer (2): Filter books using WHERE author LIKE 'A%'
SQL Query:
SELECT
    title,
    author,
    'A' AS selected_letter
FROM books
WHERE author LIKE 'A%';

Result Output:
         title          author selected_letter
As I Lay Dying  William Faulkner               A
   Animal Farm   George Orwell               A


Question (3): Get the top 5 most checked-out books and their checkout count.
Answer (3): JOIN books with checkouts, GROUP BY book, ORDER BY count DESC, LIMIT 5
SQL Query:
SELECT
    b.book_id,
    b.title,
    COUNT(c.checkout_id) AS checkout_count
FROM checkouts c
JOIN books b ON c.book_id = b.book_id
GROUP BY b.book_id, b.title
ORDER BY checkout_count DESC
LIMIT 5;

Result Output:
book_id                  title  checkout_count
    101  To Kill a Mockingbird              45
    105                   1984              38
    102       The Great Gatsby              31
    108    Pride and Prejudice              29
    110     The Catcher in Rye              25


Question (4): Get top 10 members who checked out the most books.
Answer (4): JOIN checkouts with members, GROUP BY member_id, ORDER BY total DESC, LIMIT 10
SQL Query:
SELECT
    m.member_id,
    m.first_name || ' ' || m.last_name AS name,
    COUNT(c.checkout_id) AS total_checkouts
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
GROUP BY m.member_id, name
ORDER BY total_checkouts DESC
LIMIT 10;

Result Output:
member_id            name  total_checkouts
       44    Sarah Connor               18
       12        John Doe               16
       89     Emma Watson               15
        5     Bruce Wayne               14
       23      Clark Kent               14
       67    Diana Prince               13
       11    Peter Parker               12
       34      Tony Stark               11
       55    Steve Rogers               10
       90 Natasha Romanov               10


Question (5): Get checkouts for 'Downtown' neighborhood (skip first 10, get next 10).
Answer (5): JOIN checkouts with members, filter by neighborhood, ORDER BY date DESC, LIMIT 10 OFFSET 10
SQL Query:
SELECT
    c.*,
    m.neighborhood,
    'Downtown' AS selected_neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE LOWER(m.neighborhood) = 'downtown'
ORDER BY c.checkout_date DESC
LIMIT 10 OFFSET 10;

Result Output:
Empty DataFrame (0 rows)
==================================================
"""
print(text_content)

# 31. Save the answers into the TXT file
new_filename = "31004260105918-Library-Task(1)-SQL-Answers.txt"

with open(new_filename, "w") as file:
    file.write(text_content)

from google.colab import files
files.download(new_filename)


LIBRARY DATABASE SQL QUERIES REPORT - TASK 1

Question (1): Get the total number of checkouts for each member (including members with zero checkouts).
Answer (1): LEFT JOIN members with checkouts, GROUP BY member_id
SQL Query:
SELECT 
    m.member_id,
    m.first_name || ' ' || m.last_name AS name,
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, name;

Result Output:
member_id             name  total_checkouts
        1    Alice Johnson                5
        2        Bob Smith                0
        3    Charlie Brown               12


Question (2): Get book titles where the author's name starts with 'A' (showing the selected letter).
Answer (2): Filter books using WHERE author LIKE 'A%'
SQL Query:
SELECT 
    title,
    author,
    'A' AS selected_letter
FROM books
WHERE author LIKE 'A%';

Result Output:
         title          author selected_letter
As I Lay Dying  William Faulkner             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Task (2): Data Integrity: Part(1):**

## **Data Exploration/Inspection:**



In [720]:
# 33. load the combined dataset from task 1
file_path = "31004260105918-Library-Task(1)-Combined-Data.csv"
df_task2 = pd.read_csv(file_path)

In [721]:
# 34.Get column names / see the task I'm working on is right in front of me, and I know exactly what I’m working on before I start.
print("The Names of columns of the table of the DataFrame")
print(df_task2.columns)

The Names of columns of the table of the DataFrame
Index(['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date',
       'neighborhood', 'membership_status', 'genre', 'pages',
       'publication_year', 'publisher', 'first_name', 'last_name', 'grade',
       'join_date'],
      dtype='object')


In [722]:
# 35. get a look at or an idea of ​​the DataFrame before starting work
# (First 15 rows)
print("Data Loaded From CSV file:")
print(df_task2.head(15))

Data Loaded From CSV file:
   checkout_id  member_id  book_id checkout_date return_date neighborhood  \
0         9263       1047      517    2024-10-21  2024-11-07   Heliopolis   
1         9340       1072      513    2025-08-24  2025-09-01      Zamalek   
2         9231       1053      523    2024-02-04  2024-02-16   Heliopolis   
3         9129       1032      513    2025-06-21  2025-06-29    Nasr City   
4         9370       1079      511    2025-11-11  2025-12-03       Shubra   
5         9082       1010      528    2025-11-11  2025-11-18       Maadi    
6         9238       1057      513    2024-03-28  2024-04-08   Heliopolis   
7         9012       1010      501    2025-02-17         NaN       Maadi    
8         9127       1024      506    2025-06-13         NaN    Nasr City   
9         9020       1016      506    2024-04-15  2024-05-04        Maadi   
10        9208       1044      525    2024-07-10  2024-08-01   Heliopolis   
11        9319       1065      515    2025-02-10 

In [723]:
# 36. Get dimensions \ To show the DataFrame size (number of rows and columns) \ How many rows and columns?
print("Shape(Size)of the dataframe:")
print(df_task2.shape)

Shape(Size)of the dataframe:
(417, 15)


In [724]:
# 37. Get DataFrame structure and Metadata / To find out the table structure of the DataFrame
print("Information of the dataframe:")
print(df_task2.info())

Information of the dataframe:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   checkout_id        417 non-null    object 
 1   member_id          417 non-null    int64  
 2   book_id            417 non-null    int64  
 3   checkout_date      417 non-null    object 
 4   return_date        326 non-null    object 
 5   neighborhood       412 non-null    object 
 6   membership_status  412 non-null    object 
 7   genre              417 non-null    object 
 8   pages              417 non-null    int64  
 9   publication_year   382 non-null    float64
 10  publisher          417 non-null    object 
 11  first_name         21 non-null     object 
 12  last_name          21 non-null     object 
 13  grade              21 non-null     float64
 14  join_date          20 non-null     object 
dtypes: float64(2), int64(3), object(10)
memory u

In [725]:
# 38. Get STATISTICAL SUMMARY /Get the DataFrame's ===STATISTICAL SUMMARY=== (Digital statistics) to find errors before Fixing the DataFrame
print("Statistical Summary of the dataframe:")
print(df_task2.describe())


Statistical Summary of the dataframe:
         member_id     book_id       pages  publication_year      grade
count   417.000000  417.000000  417.000000        382.000000  21.000000
mean   1041.004796  513.443645  202.299760       2017.426702   7.285714
std      26.356925    8.976053   73.986738          4.567214   1.230563
min    1001.000000  501.000000  104.000000       2009.000000   6.000000
25%    1020.000000  507.000000  134.000000       2014.000000   6.000000
50%    1036.000000  513.000000  160.000000       2017.000000   7.000000
75%    1061.000000  519.000000  294.000000       2021.000000   9.000000
max    1201.000000  532.000000  338.000000       2024.000000   9.000000


In [726]:
# 39. Check if any column has any unique values in the table
print(df_task2.nunique() == len(df_task2))

# Explore The Unique Columns (member_id and isbn):

# 1. Get the number of unique values in each column
print("Unique values count per column:")
print(df_task2.nunique())

# 2. Show the list of unique values for a specific column (e.g., member_id)
print("Unique Member IDs:")
print(df_task2['member_id'].unique())

# 3. Get unique values and their frequencies
print("Value counts for member_id:")
print(df_task2['member_id'].value_counts())

checkout_id          False
member_id            False
book_id              False
checkout_date        False
return_date          False
neighborhood         False
membership_status    False
genre                False
pages                False
publication_year     False
publisher            False
first_name           False
last_name            False
grade                False
join_date            False
dtype: bool
Unique values count per column:
checkout_id          409
member_id             68
book_id               32
checkout_date        288
return_date          245
neighborhood           9
membership_status      4
genre                  8
pages                 30
publication_year      14
publisher              4
first_name            16
last_name             11
grade                  4
join_date             17
dtype: int64
Unique Member IDs:
[1047 1072 1053 1032 1079 1010 1057 1024 1016 1044 1065 1018 1041 1070
 1034 1027 1008 1076 1059 1030 1040 1050 1020 1052 1077 1068 1022 1061
 1

In [727]:
# 40. Get a semi complete overview of the DataFrame
print(df_task2)

    checkout_id  member_id  book_id checkout_date return_date neighborhood  \
0          9263       1047      517    2024-10-21  2024-11-07   Heliopolis   
1          9340       1072      513    2025-08-24  2025-09-01      Zamalek   
2          9231       1053      523    2024-02-04  2024-02-16   Heliopolis   
3          9129       1032      513    2025-06-21  2025-06-29    Nasr City   
4          9370       1079      511    2025-11-11  2025-12-03       Shubra   
..          ...        ...      ...           ...         ...          ...   
412      WEB_22       1003      501    2025-07-08         NaN        Maadi   
413      WEB_23       1017      507    2025-07-11         NaN        Maadi   
414      WEB_24       1061      504    2025-07-06         NaN      zamalek   
415      WEB_25       1201      523    2025-07-08         NaN          NaN   
416      WEB_26       1041      512    2025-07-06         NaN    Nasr City   

    membership_status       genre  pages  publication_year  \
0

# **Task (2): Data Integrity: Part(2):**

# **Data Cleaning((Fixing)(processing)(Scaning)):**


In [728]:
# 41.Check column names safely before cleaning ISBN
isbn_col = [col for col in df_task2.columns if 'isbn' in col.lower()]

if isbn_col:
    col_name = isbn_col[0]
    df_task2[col_name] = df_task2[col_name].astype(str).str.replace('-', '')
    df_task2.loc[~df_task2[col_name].str.isdigit(), col_name] = 'Invalid_ISBN'
    print(f"ISBN cleaning done on column: {col_name}")
else:
    print("ISBN column not found. Available columns:", df_task2.columns.tolist())

ISBN column not found. Available columns: ['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'neighborhood', 'membership_status', 'genre', 'pages', 'publication_year', 'publisher', 'first_name', 'last_name', 'grade', 'join_date']


In [729]:
# 42. Handle missing values in member_id and other text columns

# Fill missing member_id with 'Unknown'
df_task2['member_id'] = df_task2['member_id'].fillna('Unknown')

print("Missing values after filling:")
print(df_task2.isna().sum())

Missing values after filling:
checkout_id            0
member_id              0
book_id                0
checkout_date          0
return_date           91
neighborhood           5
membership_status      5
genre                  0
pages                  0
publication_year      35
publisher              0
first_name           396
last_name            396
grade                396
join_date            397
dtype: int64


In [730]:
# 43. Check and drop duplicated rows

# 1. Count duplicates before deletion
print("Total duplicate rows:", df_task2.duplicated().sum())

# 2. Drop duplicates and keep the first occurrence
df_task2 = df_task2.drop_duplicates(keep='first')

print("Shape of DataFrame after removing duplicates:", df_task2.shape)

Total duplicate rows: 8
Shape of DataFrame after removing duplicates: (409, 15)


In [731]:
# 44. Fix text formatting (Remove extra spaces and capitalize text)

# Remove spaces and fix capitalization for string columns
for col in df_task2.select_dtypes(include='object').columns:
    df_task2[col] = df_task2[col].astype(str).str.strip().str.title()

print("Text formatting fixed successfully!")

Text formatting fixed successfully!


In [732]:
# 45. Save the cleaned dataset to CSV and download it (final outpu)

# 1. Save cleaned data to a new CSV file ()
clean_filename = "31004260105918-Library-Task(2)-Cleaned-Data.csv"
df_task2.to_csv(clean_filename, index=False)

# 2. Download file directly
from google.colab import files
files.download(clean_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Task (3): Data Fairness and Version Control**

In [733]:
# 46.Load the cleaned dataset using the exact Task 2 file name
file_name = "31004260105918-Library-Task(2)-Cleaned-Data.csv"
df_task2 = pd.read_csv(file_name)

# 1.Group by neighborhood to count members and checkouts side by side
summary = df_task2.groupby("neighborhood").agg(
    total_members=("member_id", "nunique"),
    total_checkouts=("checkout_id", "count")
).reset_index()

# 2.Calculate percentages to measure fairness accurately
summary["members_pct"] = (summary["total_members"] / summary["total_members"].sum()) * 100
summary["checkouts_pct"] = (summary["total_checkouts"] / summary["total_checkouts"].sum()) * 100

# 3.Print the comparison table
print(summary)

  neighborhood  total_members  total_checkouts  members_pct  checkouts_pct
0   Heliopolis             13               87    19.117647      21.271394
1        Maadi             20              114    29.411765      27.872861
2          Nan              3                5     4.411765       1.222494
3    Nasr City             16              101    23.529412      24.694377
4       Shubra              5               34     7.352941       8.312958
5      Zamalek             11               68    16.176471      16.625917
